# TeClass — Final mDeBERTa-v3-base Run (Google Colab)

**Task:** Telugu headline relevance classification  
**Model:** `microsoft/mdeberta-v3-base`  
**Labels:** HREL=0, MREL=1, LREL=2  
**Method:** Standard fine-tuning on headline–article pairs; no feature augmentation.  
**Tuning:** Reuses the best Optuna Trial 2 parameters; Optuna is not rerun.

This notebook saves checkpoints, the final model, predictions, and metrics to Google Drive. It uses a NEW output folder to avoid attempting to resume from the incomplete checkpoints in the earlier run.


In [20]:
# CELL 1 — Mount the NEW Google account's Drive
from google.colab import drive
drive.mount("/content/drive")

import os, json, gc, warnings, shutil
from pathlib import Path

DRIVE_ROOT = (
    "/content/drive/MyDrive/"
    "TeClass_mDeBERTa_Final_Run_New"
)
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "training_checkpoints")
FINAL_DIR = os.path.join(DRIVE_ROOT, "final_model")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("Saving to:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/TeClass_mDeBERTa_Final_Run_New


In [ ]:
# CELL 2 — Install compatible packages
%pip -q install "transformers==4.46.3" "datasets==3.1.0" "accelerate>=1.1.0" sentencepiece scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.

In [ ]:
# CELL 3 — Imports and GPU check
import os, json, gc, warnings
import numpy as np
import pandas as pd
import torch
import transformers, datasets, accelerate
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


Transformers: 4.46.3
Datasets: 3.1.0
Accelerate: 1.15.0
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
# CELL 4 — Configuration and best previously selected Optuna parameters
MODEL_NAME = "microsoft/mdeberta-v3-base"
MAX_LEN = 512
SEED = 42
TRAIN_BATCH = 4
EVAL_BATCH = 8
GRAD_ACCUM = 2
FINAL_EPOCHS = 4

# Best parameters from completed Optuna Trial 2 — do NOT rerun Optuna.
best_params = {
    "learning_rate": 1.405506534837736e-05,
    "weight_decay": 0.04320497666296414,
    "dropout": 0.11062038588154888,
    "warmup_ratio": 0.08543326321134513,
}

# Save all artifacts to the NEW account's Drive
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "training_checkpoints")
FINAL_DIR = os.path.join(DRIVE_ROOT, "final_model")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

with open(os.path.join(DRIVE_ROOT, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2)

In [6]:
# CELL 5 — Find and load the three CSV files from Colab or Google Drive
EXPECTED = {
    "train": "TeClass_training_data.csv",
    "dev": "TeClass_development_data.csv",
    "test": "TeClass_testing_data.csv",
}

def find_csv(filename):
    candidates = [Path("/content") / filename, Path("/content/drive/MyDrive") / filename]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    matches = list(Path("/content/drive/MyDrive").rglob(filename))
    if matches:
        return str(matches[0])
    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to Colab or put it inside Google Drive/MyDrive."
    )

TRAIN_PATH = find_csv(EXPECTED["train"])
DEV_PATH = find_csv(EXPECTED["dev"])
TEST_PATH = find_csv(EXPECTED["test"])
print("Train CSV:", TRAIN_PATH)
print("Dev CSV:  ", DEV_PATH)
print("Test CSV: ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(DEV_PATH)
test_df = pd.read_csv(TEST_PATH)
required_columns = {"article", "headline", "label"}
for split_name, df in [("train", train_df), ("development", val_df), ("test", test_df)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{split_name} CSV missing columns: {missing}")
    df["headline"] = df["headline"].fillna("").astype(str)
    df["article"] = df["article"].fillna("").astype(str)
print("Train:", train_df.shape, "Development:", val_df.shape, "Test:", test_df.shape)
display(train_df.head(3))


Train CSV: /content/TeClass_training_data.csv
Dev CSV:   /content/TeClass_development_data.csv
Test CSV:  /content/TeClass_testing_data.csv
Train: (18324, 5) Development: (3927, 5) Test: (3927, 5)


,id,url,article,headline,label
0,training_1,https://www.manatelangana.news/one-person-watc...,హైదరాబాద్ : “రాజు గారి గది” సినిమా చూస్తూ ఓ ప్...,వ్యక్తి ప్రాణాలు తీసిన “రాజు గారి గది” సినిమా,MREL
1,training_2,http://www.andhrabhoomi.net/content/business-1477,"హైదరాబాద్ , ఫిబ్రవరి 6 : బ్యాంకులు తమ బ్యాలెన్...",‘ఎన్పిఎల బాధ్యత బ్యాంకులదే’,MREL
2,training_3,https://10tv.in/movies/sekhar-kammula-father-p...,ప్రముఖ దర్శకులు శేఖర్ కమ్ముల ఇంట విషాదం నెలకొం...,శేఖర్ కమ్ముల తండ్రి శేషయ్య కన్నుమూత,HREL


In [7]:
# CELL 6 — Explicit, consistent label mapping (the main label fix)
label2id = {"HREL": 0, "MREL": 1, "LREL": 2}
id2label = {v: k for k, v in label2id.items()}
for split_name, df in [("train", train_df), ("development", val_df), ("test", test_df)]:
    original_labels = set(df["label"].dropna().astype(str).str.strip().str.upper().unique())
    if original_labels != set(label2id):
        raise ValueError(f"Unexpected labels in {split_name}: {original_labels}. Expected {set(label2id)}")
    df["label"] = df["label"].astype(str).str.strip().str.upper().map(label2id).astype(int)

print("Verified mapping:", label2id)
print("Train counts:", train_df["label"].value_counts().sort_index().to_dict())
print("Dev counts:", val_df["label"].value_counts().sort_index().to_dict())
print("Test counts:", test_df["label"].value_counts().sort_index().to_dict())


Verified mapping: {'HREL': 0, 'MREL': 1, 'LREL': 2}
Train counts: {0: 5962, 1: 7105, 2: 5257}
Dev counts: {0: 1277, 1: 1523, 2: 1127}
Test counts: {0: 1278, 1: 1523, 2: 1126}


In [8]:
# CELL 7 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Tokenizer: DebertaV2Tokenizer
Vocabulary size: 250102


In [9]:
# CELL 8 — Tokenize headline + article pairs
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

def tokenize_batch(batch):
    return tokenizer(
        batch["headline"],
        batch["article"],
        truncation=True,
        max_length=MAX_LEN,
    )

train_ds = train_ds.map(tokenize_batch, batched=True, desc="Tokenizing train")
val_ds = val_ds.map(tokenize_batch, batched=True, desc="Tokenizing development")
test_ds = test_ds.map(tokenize_batch, batched=True, desc="Tokenizing test")

print("Tokenization complete.")

Tokenizing train:   0%|          | 0/18324 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Tokenizing development:   0%|          | 0/3927 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Tokenizing test:   0%|          | 0/3927 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Tokenization complete.


In [10]:
# CELL 9 — Keep only model inputs and labels
keep_columns = ["input_ids", "attention_mask", "label"]

train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c not in keep_columns]
)
val_ds = val_ds.remove_columns(
    [c for c in val_ds.column_names if c not in keep_columns]
)
test_ds = test_ds.remove_columns(
    [c for c in test_ds.column_names if c not in keep_columns]
)

train_ds.set_format(type="torch")
val_ds.set_format(type="torch")
test_ds.set_format(type="torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Train columns:", train_ds.column_names)
print("Example keys:", train_ds[0].keys())

Train columns: ['label', 'input_ids', 'attention_mask']
Example keys: dict_keys(['label', 'input_ids', 'attention_mask'])


In [11]:
# CELL 10 — Evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

print("Metrics ready.")

Metrics ready.


In [12]:
# CELL 11 — Build standard mDeBERTa classifier (no feature augmentation)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    hidden_dropout_prob=best_params["dropout"],
    attention_probs_dropout_prob=best_params["dropout"],
    id2label=id2label,
    label2id=label2id,
)

print("Standard mDeBERTa-v3-base sequence classifier loaded.")

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Standard mDeBERTa-v3-base sequence classifier loaded.


In [13]:
# CELL 12 — Training arguments with Drive-persisted checkpoints

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    warmup_ratio=best_params["warmup_ratio"],
    num_train_epochs=FINAL_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    save_safetensors=True,
    save_only_model=False,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=250,
    report_to="none",
    fp16=True,
    bf16=False,
    seed=SEED,
    data_seed=SEED,
    optim="adamw_torch",
)

print("Training arguments ready.")

Training arguments ready.


In [14]:
# CELL 13 — Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer ready.")

Trainer ready.


In [16]:
# CELL 14 — Train or resume from the latest valid checkpoint

from transformers.trainer_utils import get_last_checkpoint
import os

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print("Resuming from checkpoint:", last_checkpoint)
    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("No checkpoint found. Starting fresh.")
    train_result = trainer.train()

print("Training finished.")

No checkpoint found. Starting fresh.


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
500,0.880800,1.003559,0.544181,0.533564,0.523523
1000,0.849300,0.917175,0.594601,0.591822,0.587206
1500,0.880700,0.878281,0.601986,0.597769,0.590278
2000,0.835800,0.824458,0.614719,0.616263,0.609798
2500,0.780800,0.858997,0.623631,0.625341,0.623610
3000,0.784100,0.870098,0.615228,0.614652,0.607821
3500,0.774100,0.862174,0.610644,0.607853,0.600439
4000,0.781000,0.870500,0.612681,0.609917,0.608533
4500,0.770500,0.923673,0.607334,0.603271,0.597679
5000,0.694800,0.885671,0.622103,0.623714,0.621384


Training finished.


In [17]:
# CELL 15 — Development evaluation
dev_results = trainer.evaluate(eval_dataset=val_ds)

print("DEVELOPMENT RESULTS")
for key, value in dev_results.items():
    if isinstance(value, (int, float, np.number)):
        print(f"{key}: {value:.6f}" if isinstance(value, (float, np.floating)) else f"{key}: {value}")

DEVELOPMENT RESULTS
eval_loss: 0.858997
eval_accuracy: 0.623631
eval_f1_macro: 0.625341
eval_f1_weighted: 0.623610
eval_runtime: 81.500300
eval_samples_per_second: 48.184000
eval_steps_per_second: 6.025000
epoch: 3.999127


In [18]:
# CELL 16 — Evaluate once on the held-out test split
test_results = trainer.evaluate(eval_dataset=test_ds)

pred_output = trainer.predict(test_ds)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = test_df["label"].to_numpy()

print("\nFINAL TEST RESULTS")
print(f"Accuracy:    {test_results['eval_accuracy']:.4f}")
print(f"Macro-F1:    {test_results['eval_f1_macro']:.4f}")
print(f"Weighted-F1: {test_results['eval_f1_weighted']:.4f}")

report = classification_report(y_true, y_pred, digits=4, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print("\nClassification report:\n", report)
print("Confusion matrix:\n", cm)


FINAL TEST RESULTS
Accuracy:    0.6193
Macro-F1:    0.6219
Weighted-F1: 0.6196

Classification report:
               precision    recall  f1-score   support

           0     0.6484    0.6854    0.6664      1278
           1     0.5540    0.6126    0.5819      1523
           2     0.6984    0.5533    0.6174      1126

    accuracy                         0.6193      3927
   macro avg     0.6336    0.6171    0.6219      3927
weighted avg     0.6262    0.6193    0.6196      3927

Confusion matrix:
 [[876 342  60]
 [381 933 209]
 [ 94 409 623]]


In [19]:
# CELL 17 — Save and verify model weights, tokenizer, metrics, and predictions
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

summary = {
    "model": MODEL_NAME,
    "max_length": MAX_LEN,
    "seed": SEED,
    "epochs_configured": FINAL_EPOCHS,
    "hyperparameters": best_params,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "development_metrics": {k: float(v) for k, v in dev_results.items() if isinstance(v, (float, np.floating, int, np.integer))},
    "test_metrics": {k: float(v) for k, v in test_results.items() if isinstance(v, (float, np.floating, int, np.integer))},
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
}
with open(os.path.join(DRIVE_ROOT, "metrics_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
pd.DataFrame({"true_label": y_true, "predicted_label": y_pred}).to_csv(os.path.join(DRIVE_ROOT, "test_predictions.csv"), index=False)
with open(os.path.join(DRIVE_ROOT, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report + "\n\nConfusion matrix:\n" + np.array2string(cm))

weight_files = [os.path.join(FINAL_DIR, name) for name in ["model.safetensors", "pytorch_model.bin"] if os.path.isfile(os.path.join(FINAL_DIR, name))]
if not weight_files:
    raise RuntimeError(f"No model weights found in {FINAL_DIR}; verify Drive is mounted and don't end the session yet.")
print("Verified model weights:")
for path in weight_files:
    print(path, f"({os.path.getsize(path):,} bytes)")
print("All artifacts saved under:", DRIVE_ROOT)
print("Test accuracy:", round(test_results["eval_accuracy"], 4))
print("Test Macro-F1:", round(test_results["eval_f1_macro"], 4))
print("Test Weighted-F1:", round(test_results["eval_f1_weighted"], 4))


Verified model weights:
/content/drive/MyDrive/TeClass_mDeBERTa_Final_Run_New/final_model/model.safetensors (1,115,271,284 bytes)
All artifacts saved under: /content/drive/MyDrive/TeClass_mDeBERTa_Final_Run_New
Test accuracy: 0.6193
Test Macro-F1: 0.6219
Test Weighted-F1: 0.6196
